# Árboles de decisión, bagging y bosques aleatorios (solución)

Este cuaderno implementa un flujo completo con prioridad en trazabilidad del experimento.


## 1) Importación de módulos


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.datasets import load_breast_cancer


## 2) Resolución de rutas de base de datos

La lógica sigue este orden:

1. Si estamos en Colab, clonar el repo de notebooks si no existe aún.
2. Buscar carpeta `Bases de datos` en rutas candidatas.
3. Cargar `spambase.data` si est? disponible.
4. Si no est? disponible, activar respaldo con `breast_cancer`.


In [ ]:
REPO_NOTEBOOKS = "https://github.com/AdanReyes2407/Notebooks_NM_ML.git"


def is_colab():
    return "google.colab" in sys.modules


def ensure_repo_clone(repo_url=REPO_NOTEBOOKS, target=Path('/content/Notebooks_NM_ML')):
    if is_colab() and not target.exists():
        print(f"Clonando repositorio en {target} ...")
        subprocess.run(["git", "clone", repo_url, str(target)], check=True)


def resolve_data_dir():
    ensure_repo_clone()
    candidates = [
        Path('/content/Notebooks_NM_ML/Bases de datos'),
        Path.cwd() / 'Bases de datos',
        Path.cwd().parent / 'Bases de datos',
        Path('/content/Bases de datos'),
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


DATA_DIR = resolve_data_dir()
print("DATA_DIR:", DATA_DIR)


In [ ]:
def load_dataset(data_dir):
    if data_dir is not None:
        spam_path = data_dir / 'spambase.data'
        if spam_path.exists():
            df = pd.read_csv(spam_path, header=None)
            X = df.iloc[:, :-1].copy()
            y = df.iloc[:, -1].astype(int).copy()
            return X, y, "spambase.data"

    bc = load_breast_cancer(as_frame=True)
    X = bc.data.copy()
    y = bc.target.astype(int).copy()
    return X, y, "sklearn_breast_cancer"


X, y, fuente = load_dataset(DATA_DIR)
print("Fuente de datos:", fuente)
print("Dimensión X:", X.shape)
print("Distribución de clase:")
print(y.value_counts(normalize=True).sort_index())


## 3) Partición train/test


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "Test:", X_test.shape)


## 4) ?rbol CART base

Se usa un ?rbol con control de profundidad para evitar crecimiento desmedido.


In [ ]:
tree = DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42)
tree.fit(X_train, y_train)
y_tree = tree.predict(X_test)

acc_tree = accuracy_score(y_test, y_tree)
print(f"Accuracy CART: {acc_tree:.4f}")
print(classification_report(y_test, y_tree, digits=4))


## 5) Bagging sobre ?rbol base


In [ ]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(max_depth=8, min_samples_leaf=5, random_state=42),
    n_estimators=250,
    max_samples=0.9,
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)
bag.fit(X_train, y_train)
y_bag = bag.predict(X_test)

acc_bag = accuracy_score(y_test, y_bag)
print(f"Accuracy Bagging: {acc_bag:.4f}")
print(classification_report(y_test, y_bag, digits=4))


## 6) Barrido de `max_features` con score OOB en Random Forest

`oob_score` sirve como estimación interna del desempeño usando observaciones no incluidas en cada bootstrap.


In [ ]:
max_feats = list(range(1, X_train.shape[1] + 1))
oob_scores = []

for mf in max_feats:
    rf_tmp = RandomForestClassifier(
        n_estimators=350,
        max_features=mf,
        oob_score=True,
        n_jobs=-1,
        random_state=42,
    )
    rf_tmp.fit(X_train, y_train)
    oob_scores.append(rf_tmp.oob_score_)

best_idx = int(np.argmax(oob_scores))
best_mf = max_feats[best_idx]
print(f"Mejor max_features por OOB: {best_mf}")
print(f"OOB máximo: {oob_scores[best_idx]:.4f}")


In [ ]:
plt.figure(figsize=(8, 4.2))
plt.plot(max_feats, oob_scores, lw=1.8)
plt.axvline(best_mf, color='tab:red', ls='--', label=f'max_features ?ptimo = {best_mf}')
plt.xlabel('max_features')
plt.ylabel('OOB score')
plt.title('Selección de max_features con OOB')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


## 7) Bosque final e importancia de variables


In [ ]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_features=best_mf,
    oob_score=True,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
y_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_rf)
print(f"Accuracy RF: {acc_rf:.4f}")
print(f"OOB RF:      {rf.oob_score_:.4f}")
print(classification_report(y_test, y_rf, digits=4))


In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns if hasattr(X, 'columns') else [f'x{i}' for i in range(X.shape[1])])
imp = imp.sort_values(ascending=False).head(15)

plt.figure(figsize=(8.5, 5))
imp.sort_values().plot(kind='barh')
plt.xlabel('Importancia relativa')
plt.ylabel('Variable')
plt.title('Top 15 variables por importancia en Random Forest')
plt.tight_layout()
plt.show()


## 8) Comparativa global y matriz de confusión del mejor modelo


In [ ]:
res = pd.DataFrame([
    ("CART", acc_tree, np.nan),
    ("Bagging", acc_bag, np.nan),
    ("Random Forest", acc_rf, rf.oob_score_),
], columns=["Modelo", "Accuracy test", "OOB"])

print(res.sort_values('Accuracy test', ascending=False))

preds = {
    "CART": y_tree,
    "Bagging": y_bag,
    "Random Forest": y_rf,
}

best_model = res.sort_values('Accuracy test', ascending=False).iloc[0]['Modelo']
print("Mejor modelo por accuracy test:", best_model)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, preds[best_model], ax=ax, cmap='Blues')
ax.set_title(f'Matriz de confusión - {best_model}')
plt.tight_layout()
plt.show()


## 9) Interpretación final

Conclusiones que deben sostenerse con números del experimento:

1. Bagging disminuye la sensibilidad de un ?rbol ?nico ante cambios de muestra.
2. Random Forest añade decorrelación entre Árboles vía `max_features`.
3. El score OOB es una referencia ?til para selección de hiperparámetros antes de abrir test.
